# JACTUS: Gallery of All 18 ACTUS Contract Types

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pedronahum/JACTUS/blob/main/examples/notebooks/06_gallery_of_contracts.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-JACTUS-blue?logo=github)](https://github.com/pedronahum/JACTUS)
[![PyPI](https://img.shields.io/pypi/v/jactus)](https://pypi.org/project/jactus/)
[![License](https://img.shields.io/badge/License-Apache%202.0-blue.svg)](https://opensource.org/licenses/Apache-2.0)

**JACTUS** implements all 18 contract types from the [ACTUS](https://www.actusfrf.org/) (Algorithmic Contract Types Unified Standards) financial contract specification using [JAX](https://jax.readthedocs.io/).

This notebook demonstrates **every contract type** in a single place, from simple cash positions to composite credit derivatives.

| Category | Types | Count |
|----------|-------|-------|
| Principal Exchange | CSH, PAM, LAM, NAM, ANN, CLM, LAX | 7 |
| Non-Principal | STK, COM, UMP | 3 |
| Derivatives | FXOUT, OPTNS, FUTUR, SWPPV | 4 |
| Composite Derivatives | SWAPS, CAPFL, CEG, CEC | 4 |
| **Total** | | **18** |

In [ ]:
!pip install -q --upgrade jactus --no-deps

In [ ]:
import jactus
print(f"JACTUS version: {jactus.__version__}")

import jax.numpy as jnp
from jactus.contracts import create_contract
from jactus.core import (
    ContractAttributes, ContractType, ContractRole,
    ActusDateTime, DayCountConvention,
)
from jactus.observers import (
    ConstantRiskFactorObserver, DictRiskFactorObserver,
)

# Common dates used across examples
SD = ActusDateTime(2024, 1, 1)
IED = ActusDateTime(2024, 1, 15)


def show_events(result, max_events=10, title=None):
    """Compact event table for any contract type."""
    if title:
        print(f"\n{title}")
        print("=" * 65)
    print(f"{'Date':<12} {'Event':<6} {'Payoff':>14}  {'Notional':>14}")
    print("-" * 50)
    events = result.events
    shown = events[:max_events]
    for e in shown:
        t = e.event_time
        date_str = f"{t.year}-{t.month:02d}-{t.day:02d}"
        payoff = float(e.payoff)
        nt = float(e.state_post.nt) if e.state_post else 0.0
        payoff_s = f"${payoff:>12,.2f}" if payoff != 0 else f"{'--':>13}"
        print(f"{date_str:<12} {e.event_type.name:<6} {payoff_s}  ${nt:>12,.2f}")
    if len(events) > max_events:
        print(f"  ... ({len(events) - max_events} more events)")
    inflows = sum(float(e.payoff) for e in events if float(e.payoff) > 0)
    outflows = sum(float(e.payoff) for e in events if float(e.payoff) < 0)
    print(f"\nInflows: ${inflows:,.2f}  |  Outflows: ${outflows:,.2f}  |  Net: ${inflows + outflows:,.2f}")

---
## 1. Principal Exchange Contracts

These contracts involve the exchange of principal: a lender disburses funds (IED), the borrower makes periodic interest and/or principal payments, and the remaining principal is returned at maturity (MD).

### 1.1 CSH -- Cash

The simplest ACTUS type. A cash position with no cash flows -- it only tracks a notional balance. Used as a building block in portfolio models and as a zero-coupon placeholder.

**Real-world examples:** Escrow accounts, cash reserves, settlement accounts.

In [ ]:
csh_attrs = ContractAttributes(
    contract_id="CSH-001", contract_type=ContractType.CSH,
    contract_role=ContractRole.RPA, status_date=SD,
    notional_principal=50_000.0, currency="USD",
)
csh_result = create_contract(csh_attrs, ConstantRiskFactorObserver(0.0)).simulate()
show_events(csh_result, title="CSH: $50K Cash Position")

### 1.2 PAM -- Principal at Maturity

Interest-only loan or bond. The full principal is exchanged at inception and returned at maturity. Periodic interest payments in between -- no amortization.

**Real-world examples:** Treasury bonds, corporate bonds, bullet loans.

In [ ]:
pam_attrs = ContractAttributes(
    contract_id="PAM-001", contract_type=ContractType.PAM,
    contract_role=ContractRole.RPA, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2027, 1, 15),
    notional_principal=100_000.0, nominal_interest_rate=0.05,
    interest_payment_cycle="6M", day_count_convention="30E360",
    currency="USD",
)
pam_result = create_contract(pam_attrs, ConstantRiskFactorObserver(0.0)).simulate()
show_events(pam_result, title="PAM: $100K Bond at 5%, Semi-Annual, 3 Years")

### 1.3 LAM -- Linear Amortizer

Fixed principal repayments reduce the outstanding balance each period. Interest is calculated on the declining balance, so total payments decrease over time.

**Real-world examples:** Auto loans, equipment financing, student loans.

In [ ]:
lam_attrs = ContractAttributes(
    contract_id="LAM-001", contract_type=ContractType.LAM,
    contract_role=ContractRole.RPA, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2026, 1, 15),
    notional_principal=60_000.0, nominal_interest_rate=0.05,
    principal_redemption_cycle="6M", next_principal_redemption_amount=15_000.0,
    principal_redemption_anchor=ActusDateTime(2024, 7, 15),
    interest_payment_cycle="6M", interest_calculation_base="NT",
    day_count_convention=DayCountConvention.A360, currency="USD",
)
lam_result = create_contract(lam_attrs, ConstantRiskFactorObserver(0.05)).simulate()
show_events(lam_result, title="LAM: $60K Auto Loan at 5%, Semi-Annual, 2 Years")

### 1.4 NAM -- Negative Amortizer

Like LAM, but if the scheduled payment is less than accrued interest, the shortfall is *added* to the principal -- the balance can **grow** over time (negative amortization).

**Real-world examples:** Payment-option ARMs, deferred-interest student loans.

In [ ]:
nam_attrs = ContractAttributes(
    contract_id="NAM-001", contract_type=ContractType.NAM,
    contract_role=ContractRole.RPA, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2027, 1, 15),
    notional_principal=100_000.0, nominal_interest_rate=0.08,
    principal_redemption_cycle="3M",
    next_principal_redemption_amount=500.0,  # Low payment -> negative amortization
    principal_redemption_anchor=ActusDateTime(2024, 4, 15),
    interest_payment_cycle="3M", interest_calculation_base="NT",
    day_count_convention=DayCountConvention.A360, currency="USD",
)
nam_result = create_contract(nam_attrs, ConstantRiskFactorObserver(0.08)).simulate()
show_events(nam_result, max_events=12, title="NAM: $100K at 8%, $500/Quarter (Negative Amortization)")

### 1.5 ANN -- Annuity (Mortgage)

Equal total payments throughout the contract life. Early payments are mostly interest; later payments are mostly principal. The classic mortgage structure.

**Real-world examples:** Home mortgages, fixed-rate personal loans.

In [ ]:
ann_attrs = ContractAttributes(
    contract_id="ANN-001", contract_type=ContractType.ANN,
    contract_role=ContractRole.RPA, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2029, 1, 15),
    notional_principal=200_000.0, nominal_interest_rate=0.06,
    interest_payment_cycle="1M", principal_redemption_cycle="1M",
    day_count_convention="A360", currency="USD",
)
ann_result = create_contract(ann_attrs, ConstantRiskFactorObserver(0.06)).simulate()
show_events(ann_result, max_events=8, title="ANN: $200K Mortgage at 6%, Monthly, 5 Years")

### Comparison: How Does the Principal Evolve?

PAM keeps the full principal until maturity. LAM reduces it in equal steps. ANN reduces it gradually, accelerating toward the end.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

for label, result, color in [
    ("PAM (interest-only)", pam_result, "navy"),
    ("LAM (linear amortization)", lam_result, "green"),
    ("ANN (annuity/mortgage)", ann_result, "crimson"),
]:
    balances = [float(e.state_post.nt) for e in result.events if e.state_post]
    ax.plot(range(len(balances)), balances, label=label, color=color, linewidth=2)

ax.set_xlabel("Event Sequence")
ax.set_ylabel("Outstanding Notional ($)")
ax.set_title("Principal Balance: PAM vs LAM vs ANN")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 1.6 CLM -- Call Money

An on-demand loan with no fixed amortization schedule. Interest accrues and principal is repaid when "called" or at an observed maturity date.

**Real-world examples:** Overnight loans, lines of credit, margin lending.

In [ ]:
clm_attrs = ContractAttributes(
    contract_id="CLM-001", contract_type=ContractType.CLM,
    contract_role=ContractRole.RPA, status_date=SD,
    initial_exchange_date=IED,
    maturity_date=ActusDateTime(2025, 1, 15),
    notional_principal=50_000.0, nominal_interest_rate=0.07,
    day_count_convention=DayCountConvention.A360, currency="USD",
)
clm_result = create_contract(clm_attrs, ConstantRiskFactorObserver(0.07)).simulate()
show_events(clm_result, title="CLM: $50K Line of Credit at 7%, Called After 1 Year")

### 1.7 LAX -- Exotic Linear Amortizer

The most flexible amortizing contract. Uses **array schedules** -- multiple sub-periods with different principal amounts and cycles. Models step-up/step-down and graduated repayment structures.

**Real-world examples:** Graduated payment mortgages, infrastructure project finance.

In [ ]:
lax_attrs = ContractAttributes(
    contract_id="LAX-001", contract_type=ContractType.LAX,
    contract_role=ContractRole.RPA, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2027, 1, 15),
    notional_principal=120_000.0, nominal_interest_rate=0.05,
    day_count_convention=DayCountConvention.A360, currency="USD",
    # Array schedules: Year 1 low, Year 2 medium, Year 3 high
    array_pr_anchor=[
        ActusDateTime(2024, 4, 15),
        ActusDateTime(2025, 1, 15),
        ActusDateTime(2026, 1, 15),
    ],
    array_pr_cycle=["3M", "3M", "3M"],
    array_pr_next=[5_000.0, 10_000.0, 15_000.0],
    array_increase_decrease=["DEC", "DEC", "DEC"],
    interest_payment_cycle="3M",
)
lax_result = create_contract(lax_attrs, ConstantRiskFactorObserver(0.05)).simulate()
show_events(lax_result, max_events=12, title="LAX: $120K Step-Up Loan (5K -> 10K -> 15K/Quarter)")

---
## 2. Non-Principal Contracts

These contracts do not involve traditional principal exchange. They represent positions in assets (stocks, commodities) or open-ended funding profiles.

### 2.1 STK -- Stock

Represents an equity position with a purchase and termination event. Payoffs reflect the buy/sell prices.

**Real-world examples:** Equity positions, ETFs, index shares.

In [ ]:
stk_attrs = ContractAttributes(
    contract_id="STK-001", contract_type=ContractType.STK,
    contract_role=ContractRole.RPA, status_date=SD, currency="USD",
    purchase_date=ActusDateTime(2024, 1, 15),
    termination_date=ActusDateTime(2025, 1, 15),
    price_at_purchase_date=100.0,
    price_at_termination_date=120.0,
)
stk_result = create_contract(stk_attrs, ConstantRiskFactorObserver(0.0)).simulate()
show_events(stk_result, title="STK: Buy at $100, Sell at $120 (20% Return)")

### 2.2 COM -- Commodity

Like STK but for physical or financial commodities. Tracks a position by quantity and price.

**Real-world examples:** Gold holdings, oil contracts, agricultural commodities.

In [ ]:
com_attrs = ContractAttributes(
    contract_id="COM-001", contract_type=ContractType.COM,
    contract_role=ContractRole.RPA, status_date=SD, currency="USD",
    purchase_date=ActusDateTime(2024, 1, 15),
    termination_date=ActusDateTime(2025, 1, 15),
    price_at_purchase_date=2_000.0,
    price_at_termination_date=2_300.0,
)
com_result = create_contract(com_attrs, ConstantRiskFactorObserver(0.0)).simulate()
show_events(com_result, title="COM: Gold Position (Buy $2,000, Sell $2,300)")

### 2.3 UMP -- Undefined Maturity Profile

All principal changes come from *observed events* rather than a fixed schedule. No fixed amortization. Interest accrues on the outstanding balance.

**Real-world examples:** Savings accounts, checking accounts, revolving credit facilities.

In [ ]:
ump_attrs = ContractAttributes(
    contract_id="UMP-001", contract_type=ContractType.UMP,
    contract_role=ContractRole.RPA, status_date=SD,
    initial_exchange_date=IED,
    maturity_date=ActusDateTime(2025, 1, 15),
    notional_principal=100_000.0, nominal_interest_rate=0.04,
    day_count_convention=DayCountConvention.A360, currency="USD",
)
ump_result = create_contract(ump_attrs, ConstantRiskFactorObserver(0.04)).simulate()
show_events(ump_result, title="UMP: $100K Demand Deposit at 4%")

---
## 3. Derivative Contracts

Derivatives derive their value from an underlier or reference rate. JACTUS implements four single-leg derivative types.

### 3.1 FXOUT -- FX Outright (Foreign Exchange Forward)

An agreement to exchange currencies at a future date. At maturity, the contract delivers (or settles) the two currency amounts.

**Real-world examples:** FX forwards, currency hedges.

In [ ]:
fxout_attrs = ContractAttributes(
    contract_id="FX-001", contract_type=ContractType.FXOUT,
    contract_role=ContractRole.RPA, status_date=SD,
    maturity_date=ActusDateTime(2025, 1, 15),
    currency="USD", currency_2="EUR",
    notional_principal=100_000.0,     # Receive USD
    notional_principal_2=90_000.0,    # Pay EUR
    delivery_settlement="D",          # Physical delivery
)
fxout_result = create_contract(fxout_attrs, ConstantRiskFactorObserver(0.0)).simulate()
show_events(fxout_result, title="FXOUT: USD 100K vs EUR 90K, 1-Year Forward")

### 3.2 OPTNS -- Options

Gives the holder the right (not obligation) to buy (call) or sell (put) an underlier at a strike price. The option buyer pays a premium (PRD event) and receives the exercise payoff (STD event) if in-the-money.

**Real-world examples:** Stock options, commodity options, interest rate swaptions.

In [ ]:
# European Call Option: Strike $100, Spot $115, Premium $5
call_attrs = ContractAttributes(
    contract_id="CALL-100", contract_type=ContractType.OPTNS,
    contract_role=ContractRole.BUY, status_date=SD,
    maturity_date=ActusDateTime(2025, 1, 15), currency="USD",
    option_type="C", option_strike_1=100.0, option_exercise_type="E",
    purchase_date=ActusDateTime(2024, 1, 15), price_at_purchase_date=5.0,
    contract_structure="SPX",  # Underlier reference
)
spx_obs = DictRiskFactorObserver({"SPX": 115.0})
call_result = create_contract(call_attrs, spx_obs).simulate()
show_events(call_result, title="OPTNS Call: Strike=$100, Spot=$115, Premium=$5 (ITM)")

# European Put Option: Strike $100, Spot $115, Premium $3
put_attrs = ContractAttributes(
    contract_id="PUT-100", contract_type=ContractType.OPTNS,
    contract_role=ContractRole.BUY, status_date=SD,
    maturity_date=ActusDateTime(2025, 1, 15), currency="USD",
    option_type="P", option_strike_1=100.0, option_exercise_type="E",
    purchase_date=ActusDateTime(2024, 1, 15), price_at_purchase_date=3.0,
    contract_structure="SPX",
)
put_result = create_contract(put_attrs, spx_obs).simulate()
show_events(put_result, title="OPTNS Put: Strike=$100, Spot=$115, Premium=$3 (OTM)")

### Option Payoff Diagrams

In [ ]:
import numpy as np

prices = np.linspace(70, 140, 200)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Call payoff
call_payoff = np.maximum(prices - 100, 0) - 5  # Net of premium
ax1.plot(prices, call_payoff, "g-", lw=2)
ax1.axhline(0, color="gray", ls="-", alpha=0.3)
ax1.axvline(100, color="red", ls="--", alpha=0.5, label="Strike $100")
ax1.fill_between(prices, 0, call_payoff, where=call_payoff > 0, alpha=0.15, color="green")
ax1.set_title("Call Option P&L (Premium=$5)")
ax1.set_xlabel("Stock Price at Expiry ($)")
ax1.set_ylabel("P&L ($)")
ax1.legend()
ax1.grid(alpha=0.3)

# Put payoff
put_payoff = np.maximum(100 - prices, 0) - 3  # Net of premium
ax2.plot(prices, put_payoff, "r-", lw=2)
ax2.axhline(0, color="gray", ls="-", alpha=0.3)
ax2.axvline(100, color="green", ls="--", alpha=0.5, label="Strike $100")
ax2.fill_between(prices, 0, put_payoff, where=put_payoff > 0, alpha=0.15, color="red")
ax2.set_title("Put Option P&L (Premium=$3)")
ax2.set_xlabel("Stock Price at Expiry ($)")
ax2.set_ylabel("P&L ($)")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3.3 FUTUR -- Futures

An obligatory agreement to buy/sell an underlier at a future date at an agreed price. Unlike options, both parties are obligated. Payoff is linear.

**Real-world examples:** Commodity futures, bond futures, index futures.

In [ ]:
futur_attrs = ContractAttributes(
    contract_id="FUT-GC", contract_type=ContractType.FUTUR,
    contract_role=ContractRole.BUY, status_date=SD,
    maturity_date=ActusDateTime(2024, 12, 31), currency="USD",
    purchase_date=ActusDateTime(2024, 1, 15),
    price_at_purchase_date=2_200.0,
    future_price=2_200.0,
    contract_structure="GOLD",  # Underlier reference
)
gold_obs = DictRiskFactorObserver({"GOLD": 2_500.0})
futur_result = create_contract(futur_attrs, gold_obs).simulate()
show_events(futur_result, title="FUTUR: Gold 1oz, Agreed $2,200, Spot $2,500")

### 3.4 SWPPV -- Plain Vanilla Interest Rate Swap

Exchange fixed-rate payments for floating-rate payments on the same notional. No principal is exchanged -- only the net interest differential.

**Real-world examples:** Interest rate swaps, basis swaps.

In [ ]:
swppv_attrs = ContractAttributes(
    contract_id="IRS-001", contract_type=ContractType.SWPPV,
    contract_role=ContractRole.RFL,  # Receive-floating, pay-fixed
    status_date=SD,
    initial_exchange_date=IED,
    maturity_date=ActusDateTime(2027, 1, 15),
    currency="USD", notional_principal=1_000_000.0,
    nominal_interest_rate=0.03,      # Fixed leg rate
    nominal_interest_rate_2=0.04,    # Floating leg rate
    day_count_convention=DayCountConvention.A360,
    interest_payment_cycle="6M",
)
swppv_result = create_contract(swppv_attrs, ConstantRiskFactorObserver(0.0)).simulate()
show_events(swppv_result, max_events=10, title="SWPPV: $1M IRS, Fixed 3% vs Float 4%, 3Y Semi-Annual")

---
## 4. Composite Contracts

These contracts reference **child contracts**. The child is simulated first, then its results feed into the parent contract. JACTUS handles this with the `ChildContractObserver` pattern.

### 4.1 SWAPS -- Generic Swap (Composite)

A swap composed of two child legs (each a SWPPV or other type). Enables cross-currency swaps, basis swaps, and custom multi-leg structures.

**Real-world examples:** Cross-currency basis swaps, total return swaps.

In [ ]:
from jactus.observers.child_contract import SimulatedChildContractObserver

# Step 1: Define and simulate two swap legs
leg1_attrs = ContractAttributes(
    contract_id="LEG-1", contract_type=ContractType.SWPPV,
    contract_role=ContractRole.RFL, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2027, 1, 15),
    notional_principal=1_000_000.0,
    nominal_interest_rate=0.04, nominal_interest_rate_2=0.03,
    day_count_convention=DayCountConvention.A360,
    interest_payment_cycle="6M", currency="USD",
)
leg2_attrs = ContractAttributes(
    contract_id="LEG-2", contract_type=ContractType.SWPPV,
    contract_role=ContractRole.RPA, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2027, 1, 15),
    notional_principal=1_000_000.0,
    nominal_interest_rate=0.05, nominal_interest_rate_2=0.04,
    day_count_convention=DayCountConvention.A360,
    interest_payment_cycle="6M", currency="USD",
)

# Step 2: Register simulated children
swap_child_obs = SimulatedChildContractObserver()
for leg_attrs in [leg1_attrs, leg2_attrs]:
    leg = create_contract(leg_attrs, ConstantRiskFactorObserver(0.0))
    leg_result = leg.simulate()
    swap_child_obs.register_simulation(
        leg_attrs.contract_id, leg_result.events, leg_attrs, leg_result.initial_state,
    )

# Step 3: Create and simulate parent swap
swaps_attrs = ContractAttributes(
    contract_id="SWAPS-001", contract_type=ContractType.SWAPS,
    contract_role=ContractRole.RPA, status_date=SD,
    maturity_date=ActusDateTime(2027, 1, 15),
    contract_structure='{"FirstLeg": "LEG-1", "SecondLeg": "LEG-2"}',
    delivery_settlement="D", currency="USD",
)
swaps_result = create_contract(
    swaps_attrs, ConstantRiskFactorObserver(0.0), swap_child_obs
).simulate()
show_events(swaps_result, max_events=8, title="SWAPS: Two-Leg Swap (4%/3% vs 5%/4%), 3Y")

### 4.2 CAPFL -- Interest Rate Cap/Floor (Composite)

Protects against interest rate movements. A **cap** pays out when rates exceed the cap strike; a **floor** pays when rates fall below.

**Real-world examples:** Interest rate caps purchased by floating-rate borrowers, collars.

In [ ]:
# Step 1: Simulate the underlying floating-rate loan
float_loan_attrs = ContractAttributes(
    contract_id="FLOAT-LOAN", contract_type=ContractType.PAM,
    contract_role=ContractRole.RPA, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2027, 1, 15),
    notional_principal=1_000_000.0, nominal_interest_rate=0.05,
    day_count_convention=DayCountConvention.A360,
    interest_payment_cycle="3M",
    market_object_code_of_rate_reset="LIBOR-3M",
    rate_reset_cycle="3M",
)
rf_obs = ConstantRiskFactorObserver(0.06)  # Rates above cap
float_result = create_contract(float_loan_attrs, rf_obs).simulate()

# Step 2: Register as child
capfl_child_obs = SimulatedChildContractObserver()
capfl_child_obs.register_simulation(
    "FLOAT-LOAN", float_result.events, float_loan_attrs, float_result.initial_state,
)

# Step 3: Create cap
capfl_attrs = ContractAttributes(
    contract_id="CAP-001", contract_type=ContractType.CAPFL,
    contract_role=ContractRole.BUY, status_date=SD,
    maturity_date=ActusDateTime(2027, 1, 15),
    notional_principal=1_000_000.0,
    rate_reset_cap=0.05,  # 5% cap strike
    contract_structure='{"Underlying": "FLOAT-LOAN"}',
    currency="USD",
)
capfl_result = create_contract(capfl_attrs, rf_obs, capfl_child_obs).simulate()
show_events(capfl_result, title="CAPFL: $1M Interest Rate Cap at 5%")

### 4.3 CEG -- Credit Enhancement Guarantee (Composite)

A guarantee on a covered contract. If a credit event occurs on the covered loan, the guarantee pays out.

**Real-world examples:** Loan guarantees, credit default swaps, mortgage insurance.

In [ ]:
# Step 1: Simulate the covered loan
covered_loan_attrs = ContractAttributes(
    contract_id="LOAN-001", contract_type=ContractType.PAM,
    contract_role=ContractRole.RPA, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2027, 1, 15),
    notional_principal=500_000.0, nominal_interest_rate=0.04,
    day_count_convention=DayCountConvention.A360,
    interest_payment_cycle="1Y", currency="USD",
)
loan_result = create_contract(covered_loan_attrs, ConstantRiskFactorObserver(0.0)).simulate()

# Step 2: Register as child
ceg_child_obs = SimulatedChildContractObserver()
ceg_child_obs.register_simulation(
    "LOAN-001", loan_result.events, covered_loan_attrs, loan_result.initial_state,
)

# Step 3: Create guarantee
ceg_attrs = ContractAttributes(
    contract_id="CEG-001", contract_type=ContractType.CEG,
    contract_role=ContractRole.BUY, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2027, 1, 15),
    notional_principal=500_000.0,
    coverage_of_credit_enhancement=1.0,  # 100% coverage
    contract_structure='{"CoveredContract": "LOAN-001"}',
    currency="USD",
)
ceg_result = create_contract(
    ceg_attrs, ConstantRiskFactorObserver(0.0), ceg_child_obs
).simulate()
show_events(ceg_result, title="CEG: 100% Guarantee on $500K PAM Loan")

### 4.4 CEC -- Credit Enhancement Collateral (Composite)

Collateral backing a covered contract. References both the covered contract and the covering (collateral) contract.

**Real-world examples:** Collateralized loans, secured bonds, repo agreements.

In [ ]:
# Step 1: Simulate the collateral (a bond)
collateral_attrs = ContractAttributes(
    contract_id="BOND-COLL", contract_type=ContractType.PAM,
    contract_role=ContractRole.RPA, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2028, 1, 15),
    notional_principal=600_000.0, nominal_interest_rate=0.03,
    day_count_convention=DayCountConvention.A360,
    interest_payment_cycle="1Y", currency="USD",
)
coll_result = create_contract(collateral_attrs, ConstantRiskFactorObserver(0.0)).simulate()

# Step 2: Register both covered loan and collateral
cec_child_obs = SimulatedChildContractObserver()
cec_child_obs.register_simulation(
    "LOAN-001", loan_result.events, covered_loan_attrs, loan_result.initial_state,
)
cec_child_obs.register_simulation(
    "BOND-COLL", coll_result.events, collateral_attrs, coll_result.initial_state,
)

# Step 3: Create collateral enhancement
cec_attrs = ContractAttributes(
    contract_id="CEC-001", contract_type=ContractType.CEC,
    contract_role=ContractRole.BUY, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2027, 1, 15),
    notional_principal=500_000.0,
    coverage_of_credit_enhancement=0.8,  # 80% coverage
    contract_structure='{"CoveredContract": "LOAN-001", "CoveringContract": "BOND-COLL"}',
    currency="USD",
)
cec_result = create_contract(
    cec_attrs, ConstantRiskFactorObserver(0.0), cec_child_obs
).simulate()
show_events(cec_result, title="CEC: 80% Collateral ($600K Bond) on $500K Loan")

---
## 5. All 18 Types at a Glance

In [ ]:
all_types = [
    ("CSH",   "Cash",                    csh_result),
    ("PAM",   "Principal at Maturity",   pam_result),
    ("LAM",   "Linear Amortizer",        lam_result),
    ("NAM",   "Negative Amortizer",      nam_result),
    ("ANN",   "Annuity (Mortgage)",      ann_result),
    ("CLM",   "Call Money",              clm_result),
    ("LAX",   "Exotic Linear Amortizer", lax_result),
    ("STK",   "Stock",                   stk_result),
    ("COM",   "Commodity",               com_result),
    ("UMP",   "Undefined Maturity",      ump_result),
    ("FXOUT", "FX Outright",             fxout_result),
    ("OPTNS", "Options (Call)",          call_result),
    ("FUTUR", "Futures",                 futur_result),
    ("SWPPV", "Plain Vanilla Swap",      swppv_result),
    ("SWAPS", "Generic Swap",            swaps_result),
    ("CAPFL", "Cap/Floor",               capfl_result),
    ("CEG",   "Credit Guarantee",        ceg_result),
    ("CEC",   "Credit Collateral",       cec_result),
]

print(f"{'Type':<8} {'Name':<28} {'Events':>7} {'Inflows':>14} {'Outflows':>14} {'Net':>14}")
print("=" * 90)
for code_name, name, result in all_types:
    n = len(result.events)
    inf = sum(float(e.payoff) for e in result.events if float(e.payoff) > 0)
    out = sum(float(e.payoff) for e in result.events if float(e.payoff) < 0)
    net = inf + out
    print(f"{code_name:<8} {name:<28} {n:>7} ${inf:>12,.2f} ${out:>12,.2f} ${net:>12,.2f}")
print("=" * 90)
print(f"\n18 / 18 ACTUS contract types simulated successfully.")

---
## 6. Advanced Features

JACTUS goes beyond individual contract simulation. This section showcases portfolio-level features, JAX automatic differentiation, and behavioral risk models.

### 6.1 Unified Portfolio API

Simulate a mixed-type portfolio in one call. JACTUS groups contracts by type and dispatches to optimized JIT-compiled batch kernels (12 of 18 types have array-mode support).

In [ ]:
from jactus.contracts import simulate_portfolio, BATCH_SUPPORTED_TYPES

portfolio = [
    (pam_attrs, ConstantRiskFactorObserver(0.0)),
    (lam_attrs, ConstantRiskFactorObserver(0.05)),
    (ann_attrs, ConstantRiskFactorObserver(0.06)),
    (csh_attrs, ConstantRiskFactorObserver(0.0)),
    (clm_attrs, ConstantRiskFactorObserver(0.07)),
]

result = simulate_portfolio(portfolio)

print(f"Portfolio: {result['num_contracts']} contracts")
print(f"  Batch (JIT array-mode):  {result['batch_contracts']}")
print(f"  Fallback (Python):       {result['fallback_contracts']}")
print(f"  Types: {', '.join(t.value for t in result['types_used'])}")
print(f"\nPer-contract total cashflows:")
for i, (attrs, _) in enumerate(portfolio):
    cf = float(result["total_cashflows"][i])
    print(f"  {attrs.contract_id}: ${cf:>12,.2f}")

print(f"\nArray-mode supported types ({len(BATCH_SUPPORTED_TYPES)}/18):")
print(f"  {', '.join(sorted(t.value for t in BATCH_SUPPORTED_TYPES))}")

### 6.2 JAX Automatic Differentiation: Risk Metrics

Because the array-mode simulation kernel is pure JAX, we can compute exact risk sensitivities with `jax.grad`. No finite-difference bumps needed.

In [ ]:
import jax
from jactus.contracts.pam_array import precompute_pam_arrays, simulate_pam_array

# Create a mid-life bond for gradient computation
grad_attrs = ContractAttributes(
    contract_id="GRAD-001", contract_type=ContractType.PAM,
    contract_role=ContractRole.RPA,
    status_date=ActusDateTime(2025, 6, 1),
    initial_exchange_date=ActusDateTime(2022, 6, 1),
    maturity_date=ActusDateTime(2030, 6, 1),
    notional_principal=100_000.0, nominal_interest_rate=0.05,
    interest_payment_cycle="6M", day_count_convention=DayCountConvention.A360,
    currency="USD",
)
init_state, et, yf, rf, params = precompute_pam_arrays(grad_attrs, ConstantRiskFactorObserver(0.0))
_, sim_payoffs = simulate_pam_array(init_state, et, yf, rf, params)
cum_yf = jnp.cumsum(yf)


def pv_of_yield(discount_rate):
    """Present value as a differentiable function of the discount rate."""
    disc = jnp.power(1.0 + discount_rate, -cum_yf)
    return jnp.sum(sim_payoffs * disc)


# Compute risk metrics via autodiff
base_yield = jnp.float32(0.045)
pv_val = float(pv_of_yield(base_yield))
dPV_dy = float(jax.grad(pv_of_yield)(base_yield))
d2PV_dy2 = float(jax.grad(jax.grad(pv_of_yield))(base_yield))

dv01 = dPV_dy * 0.0001
mod_duration = -dPV_dy / pv_val
convexity = d2PV_dy2 / pv_val

print("Risk Metrics via JAX Automatic Differentiation")
print("=" * 55)
print(f"Coupon:            {grad_attrs.nominal_interest_rate:.2%}")
print(f"Discount rate:     {float(base_yield):.2%}")
print(f"PV:                ${pv_val:,.2f}")
print(f"DV01:              ${dv01:,.4f}")
print(f"Modified Duration: {mod_duration:.4f}")
print(f"Convexity:         {convexity:.4f}")
print("=" * 55)

### 6.3 Array-Mode Batch Simulation

For large portfolios, JACTUS uses a JIT-compiled `jax.lax.scan` kernel that processes all contracts simultaneously as `[B, T]` arrays.

In [ ]:
import random, time
from jactus.contracts.pam_array import prepare_pam_batch, batch_simulate_pam_auto

# Generate 1,000 random PAM loans
rng = random.Random(42)
batch_contracts = []
obs = ConstantRiskFactorObserver(0.0)
for i in range(1200):
    orig_year = rng.randint(2020, 2024)
    orig_month = rng.randint(1, 12)
    term = rng.randint(3, 10)
    mat = ActusDateTime(orig_year + term, orig_month, 15)
    if mat <= ActusDateTime(2025, 6, 1):
        continue
    a = ContractAttributes(
        contract_id=f"LOAN-{i:04d}", contract_type=ContractType.PAM,
        contract_role=ContractRole.RPA,
        status_date=ActusDateTime(2025, 6, 1),
        initial_exchange_date=ActusDateTime(orig_year, orig_month, 15),
        maturity_date=mat,
        notional_principal=round(rng.uniform(50_000, 500_000), -3),
        nominal_interest_rate=round(rng.uniform(0.03, 0.08), 4),
        day_count_convention=rng.choice([DayCountConvention.A360, DayCountConvention.A365]),
        interest_payment_cycle=rng.choice(["1M", "3M", "6M"]),
    )
    batch_contracts.append((a, obs))

# Prepare arrays
t0 = time.perf_counter()
states, et, yf, rf, params, masks = prepare_pam_batch(batch_contracts)
t_prep = time.perf_counter() - t0

# JIT warm-up
final_states, payoffs = batch_simulate_pam_auto(states, et, yf, rf, params)
payoffs.block_until_ready()

# Timed run
t0 = time.perf_counter()
final_states, payoffs = batch_simulate_pam_auto(states, et, yf, rf, params)
payoffs.block_until_ready()
t_kernel = time.perf_counter() - t0

print(f"Batch Simulation: {len(batch_contracts):,} PAM Loans")
print("=" * 50)
print(f"Pre-computation:  {t_prep * 1000:.1f} ms")
print(f"JIT kernel:       {t_kernel * 1000:.3f} ms")
print(f"Total:            {(t_prep + t_kernel) * 1000:.1f} ms")
print(f"Throughput:       {len(batch_contracts) / (t_prep + t_kernel):,.0f} contracts/sec")
print(f"Backend:          {jax.default_backend()}")

### 6.4 Portfolio PV Sensitivity

After batch simulation, running yield curve scenarios is nearly free -- just re-discount the pre-computed payoffs across 200 scenarios.

In [ ]:
masked_payoffs = payoffs * masks
cum_yfs = jnp.cumsum(yf, axis=1)

rates = jnp.linspace(0.01, 0.10, 200)
pvs = []
for rate in rates:
    disc = jnp.power(1.0 + float(rate), -cum_yfs)
    pvs.append(float(jnp.sum(masked_payoffs * disc)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot([float(r) * 100 for r in rates], [pv / 1e6 for pv in pvs], linewidth=2, color="navy")
ax.set_xlabel("Discount Rate (%)")
ax.set_ylabel("Portfolio PV ($M)")
ax.set_title(f"PV Sensitivity -- {len(batch_contracts):,} PAM Loans x 200 Scenarios")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.5 Behavioral Observers: Prepayment Model

JACTUS supports behavioral risk models that inject **callout events** into the simulation timeline. The `PrepaymentSurfaceObserver` uses a 2D surface (spread x loan age) to determine prepayment rates.

In [ ]:
from jactus.observers.prepayment import PrepaymentSurfaceObserver
from jactus.observers.scenario import Scenario
from jactus.utilities.surface import Surface2D

# Define a prepayment surface: rows=spread (%), cols=loan age (years)
surface = Surface2D(
    x_margins=jnp.array([-2.0, 0.0, 1.0, 2.0, 3.0]),
    y_margins=jnp.array([0.0, 1.0, 2.0, 3.0, 5.0]),
    values=jnp.array([
        [0.00, 0.00, 0.00, 0.00, 0.00],  # spread=-2%
        [0.00, 0.01, 0.01, 0.00, 0.00],  # spread= 0%
        [0.00, 0.02, 0.03, 0.01, 0.00],  # spread= 1%
        [0.01, 0.04, 0.06, 0.03, 0.01],  # spread= 2%
        [0.02, 0.06, 0.10, 0.05, 0.02],  # spread= 3%
    ]),
)

prepay_observer = PrepaymentSurfaceObserver(
    surface=surface, fixed_market_rate=0.04, prepayment_cycle="6M",
)

# Create a scenario bundling market data + behavioral model
scenario = Scenario(
    scenario_id="prepay-demo",
    description="Base case with moderate prepayment",
    market_observers={"rates": ConstantRiskFactorObserver(0.04)},
    behavior_observers={"prepayment": prepay_observer},
)

# Generate callout events for a 10-year mortgage
prepay_attrs = ContractAttributes(
    contract_id="PPY-MORTGAGE", contract_type=ContractType.ANN,
    contract_role=ContractRole.RPA, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2034, 1, 15),
    notional_principal=300_000.0, nominal_interest_rate=0.06,
    interest_payment_cycle="1M", principal_redemption_cycle="1M",
    day_count_convention="A360", currency="USD",
)
callout_events = scenario.get_callout_events(prepay_attrs)

print(f"Scenario: {scenario.scenario_id}")
print(f"Description: {scenario.description}")
print(f"Callout events generated: {len(callout_events)}")
print(f"\n{'Date':<14} {'Type':<6} {'Model ID'}")
print("-" * 45)
for ce in callout_events[:8]:
    print(f"{ce.time.year}-{ce.time.month:02d}-{ce.time.day:02d}   {ce.callout_type:<6} {ce.model_id}")
if len(callout_events) > 8:
    print(f"  ... ({len(callout_events) - 8} more events)")

---
## Next Steps

You have seen all 18 ACTUS contract types and key advanced features. To go deeper:

| Topic | Notebook |
|-------|----------|
| PAM in detail | [00 - Getting Started](https://colab.research.google.com/github/pedronahum/JACTUS/blob/main/examples/notebooks/00_getting_started_pam.ipynb) |
| Mortgage amortization | [01 - Annuity Mortgage](https://colab.research.google.com/github/pedronahum/JACTUS/blob/main/examples/notebooks/01_annuity_mortgage.ipynb) |
| Options contracts | [02 - Options](https://colab.research.google.com/github/pedronahum/JACTUS/blob/main/examples/notebooks/02_options_contracts.ipynb) |
| Interest rate caps | [03 - Interest Rate Cap](https://colab.research.google.com/github/pedronahum/JACTUS/blob/main/examples/notebooks/03_interest_rate_cap.ipynb) |
| Stocks & commodities | [04 - Stock & Commodity](https://colab.research.google.com/github/pedronahum/JACTUS/blob/main/examples/notebooks/04_stock_commodity.ipynb) |
| GPU/TPU benchmarks | [05 - GPU/TPU Benchmark](https://colab.research.google.com/github/pedronahum/JACTUS/blob/main/examples/notebooks/05_gpu_tpu_portfolio_benchmark.ipynb) |

### Resources

- **Documentation**: [pedronahum.github.io/JACTUS](https://pedronahum.github.io/JACTUS/)
- **GitHub**: [github.com/pedronahum/JACTUS](https://github.com/pedronahum/JACTUS)
- **PyPI**: [pypi.org/project/jactus](https://pypi.org/project/jactus/)
- **ACTUS Standard**: [actusfrf.org](https://www.actusfrf.org/)